# Model Dataset Creation

Step 3a of the training pipeline: joins the six processed tables from Step 1
into one table with one row per applicant (the "model dataset", MDS), keeps
the features that scored well enough in Step 2, and saves it.
`01b_multicollinearity_reduction.ipynb` then removes redundant features
from it.

**Sections**
1. Settings, including the score threshold and the always-keep list
2. Load and join the processed tables
3. Keep features that scored well enough (plus the always-keep list)
4. Check for duplicate columns
5. Mark text columns as categories
6. Quality checks and save
7. Free memory

**Outputs**
- `data/Model-Training/ModelDataset/model_dataset_initial.parquet`
- `data/Model-Training/ModelDataset/selected_features.csv` — the features kept
- `data/Model-Training/Variable-Selection/feature_exceptions.csv` — the always-keep features found

Run by the `03_mds_creation_and_multicollinearity_test` Airflow DAG (task
`create_mds`), or by hand.

## 1. Imports & Configuration

`DATA_ROOT` is the project's `data/` folder: `/data` inside Docker,
otherwise this project's folder on your PC.

- `MIN_SELECTION_POINTS` — a numeric feature is kept if its Step 2 points
  score is **above** this (scores go up to 5).
- `FEATURE_EXCEPTION_PATTERNS` — features to keep whatever their score
  (patterns matched against column names).

In [2]:
import os
import pandas as pd
import numpy as np
import gc;

DATA_ROOT = os.environ.get("HOMECREDIT_DATA_ROOT", r"a:\Dominic\My Projects\Home Credit\data")
SRC_ROOT  = os.environ.get("HOMECREDIT_SRC_DIR", r"a:\Dominic\My Projects\Home Credit\src")

VAR_SELECTION_DIR = os.path.join(DATA_ROOT, "Model-Training", "Variable-Selection")
STG_DIR = os.path.join(DATA_ROOT, "Model-Training", "Staging")           # Step 1's processed tables
MDS_DIR = os.path.join(DATA_ROOT, "Model-Training", "ModelDataset")      # output

# One row per previous loan, not per applicant -- used by other notebooks, not joined here
EXCLUDE_FROM_JOIN = {"prev_loan_data_processed.parquet"}

# Each feature's points score, from select_vars.ipynb (Step 2)
PATH_SELECTION_METRICS = os.path.join(VAR_SELECTION_DIR, "numeric_selection_metrics_full_df.xlsx")
os.makedirs(VAR_SELECTION_DIR, exist_ok=True)

# Keep numeric features scoring MORE than this (out of 5); 0–1 points is treated as noise
MIN_SELECTION_POINTS = 1

# Always keep features whose names match these patterns, whatever their score.
# They're the bureau monthly-history lateness features: 65–94% missing (most
# applicants have no bureau_balance rows), which drags their one-feature
# scores down to 0–1 points even though past lateness is one of the strongest
# signs of default. They still go through the VIF step in
# 01b_multicollinearity_reduction.ipynb, which removes redundant variants
# (MEAN vs MAX, overlapping periods).
FEATURE_EXCEPTION_PATTERNS = [
    r"^BUREAUCREDIT_.*DAYS_IN_DPD_GE1",     # months overdue
    r"^BUREAUCREDIT_.*DAYS_WITH_NO_DPD",    # months paid on time
    r"^BUREAUCREDIT_.*NUM_STATUS_CHANGES",  # how often the payment status changed
    r"^BUREAUCREDIT_.*MONTHS_TO_DPD_GE1",   # months until first overdue
]

PATH_OUTPUT_DATASET  = os.path.join(MDS_DIR, "model_dataset_initial.parquet")
PATH_OUTPUT_FEATURES = os.path.join(MDS_DIR, "selected_features.csv")
PATH_OUTPUT_EXCEPTIONS = os.path.join(VAR_SELECTION_DIR, "feature_exceptions.csv")

## 2. Load and Join the Processed Tables

Loads every parquet in `data/Model-Training/Staging/` except the loan-level
`prev_loan_data_processed.parquet`, and joins them on `SK_ID_CURR`.

To save memory (the joined table is large), each table's numbers are shrunk
on load: decimals to float32, whole numbers to the smallest type that fits.
The inference code does the same (`features/pipeline.py`), because the
trained model expects these types.

The join keeps every applicant in the **first** table loaded and adds the
others' columns (blank where an applicant has no rows). The tables are taken
in the order the folder lists them; the application table (every applicant)
comes first alphabetically, which is what keeps all applicants in.

In [3]:
def downcast_numeric(df, label=""):
    """
    Shrink number types to roughly halve memory before joining: float64 ->
    float32, int64 -> the smallest whole-number type that fits.
    SK_ID_CURR is left alone so the join keys still match.
    """
    float_cols = df.select_dtypes(include='float64').columns
    int_cols   = df.select_dtypes(include='int64').columns.difference(['SK_ID_CURR'])

    before_mb = df.memory_usage(deep=True).sum() / 1e6
    if len(float_cols) > 0:
        df[float_cols] = df[float_cols].astype('float32')
    if len(int_cols) > 0:
        df[int_cols] = df[int_cols].apply(pd.to_numeric, downcast='integer')
    after_mb = df.memory_usage(deep=True).sum() / 1e6

    print(f"  {label}: {before_mb:,.0f} MB -> {after_mb:,.0f} MB "
          f"({len(float_cols)} float cols, {len(int_cols)} int cols downcast)")
    return df


all_parquet_files = [f for f in os.listdir(STG_DIR) if f.endswith('.parquet')]
# Skip the loan-level table (no error if it isn't there)
feature_table_files = [f for f in all_parquet_files if f not in EXCLUDE_FROM_JOIN]

# Load each table, shrinking its number types straight away to save memory
dataframes = {}
for parquet_file in feature_table_files:
    file_path = os.path.join(STG_DIR, parquet_file)
    key = os.path.splitext(parquet_file)[0]
    df = pd.read_parquet(file_path)
    dataframes[key] = downcast_numeric(df, label=key)
    print(f"Loaded {key}: {dataframes[key].shape}")

  application_train_data_processed: 488 MB -> 297 MB (78 float cols, 44 int cols downcast)
Loaded application_train_data_processed: (307511, 133)
  bureau_processed: 316 MB -> 150 MB (123 float cols, 20 int cols downcast)
Loaded bureau_processed: (263491, 144)
  creditcard_processed: 505 MB -> 246 MB (676 float cols, 44 int cols downcast)
Loaded creditcard_processed: (86905, 721)
  payment_installments_processed: 324 MB -> 159 MB (118 float cols, 14 int cols downcast)
Loaded payment_installments_processed: (291643, 133)
  posdata_processed: 115 MB -> 53 MB (28 float cols, 15 int cols downcast)
Loaded posdata_processed: (289444, 44)
  prev_loans_aggregate_final_processed: 151 MB -> 74 MB (47 float cols, 11 int cols downcast)
Loaded prev_loans_aggregate_final_processed: (291057, 59)


In [4]:
# Each table must have one row per applicant -- otherwise the join below
# would quietly duplicate rows
for name, df in dataframes.items():
    n_dupes = df['SK_ID_CURR'].duplicated().sum()
    assert n_dupes == 0, f"{name} has {n_dupes} duplicate SK_ID_CURR — fix upstream before joining."
print("All tables have unique SK_ID_CURR — safe to join.")

All tables have unique SK_ID_CURR — safe to join.


In [5]:
join_keys = ['SK_ID_CURR']
print(f"Using join key(s): {join_keys}")

combined_df = None

for name, df in dataframes.items():
    df = df.set_index('SK_ID_CURR')

    if combined_df is None:
        # The first table is the base: its applicants are the ones kept
        combined_df = df
    else:
        # Joining on the index uses less memory than repeated .merge() calls
        combined_df = combined_df.join(df, how='left')

    del df
    gc.collect()   # free memory after every join
    print(f"After merging {name}: {combined_df.shape}")

combined_df = combined_df.reset_index()

print(f"\nFinal combined DataFrame shape: {combined_df.shape}")

Using join key(s): ['SK_ID_CURR']
After merging application_train_data_processed: (307511, 132)
After merging bureau_processed: (307511, 275)
After merging creditcard_processed: (307511, 995)
After merging payment_installments_processed: (307511, 1127)
After merging posdata_processed: (307511, 1170)
After merging prev_loans_aggregate_final_processed: (307511, 1228)

Final combined DataFrame shape: (307511, 1229)


In [6]:
del dataframes
gc.collect()

0

## 3. Keep the Features That Scored Well Enough

Keeps:

- numeric features scoring **more than** `MIN_SELECTION_POINTS` in Step 2
  (`select_vars.ipynb`, scores out of 5)
- features matching `FEATURE_EXCEPTION_PATTERNS` (whatever their score —
  saved to `feature_exceptions.csv`, and still subject to the VIF step next)
- every text (category) column, since Step 2 only scores numeric features
- every yes/no "had activity in this period" flag (`IS_IN_` columns)

If an exception pattern matches no column, the notebook stops — that
usually means a feature was renamed in Step 1 and the pattern needs updating.

In [7]:
numeric_selection_metrics_full_df = pd.read_excel(PATH_SELECTION_METRICS)

# Numeric features scoring above the threshold
numeric_feature_cols = (
    numeric_selection_metrics_full_df
    .loc[numeric_selection_metrics_full_df['points'] > MIN_SELECTION_POINTS, 'variable']
    .tolist()
)

# Always-keep features
import re
exception_cols = [
    col for col in combined_df.columns
    if any(re.search(p, col) for p in FEATURE_EXCEPTION_PATTERNS)
]
# A pattern matching nothing probably means a feature was renamed upstream
unmatched = [p for p in FEATURE_EXCEPTION_PATTERNS if not any(re.search(p, col) for col in exception_cols)]
assert not unmatched, f"Exception patterns matched no columns: {unmatched}"

cat_cols = combined_df.select_dtypes(include=['object', 'category']).columns.tolist()   # always kept
time_indicator_cols = [c for c in combined_df.columns  if 'IS_IN_' in c]               # always kept
#cat_cols.remove('SK_ID_CURR')

# All of the above, without duplicates, in this order
feature_cols = pd.unique(numeric_feature_cols + exception_cols + cat_cols + time_indicator_cols).tolist()
n_new_exceptions = len(set(exception_cols) - set(numeric_feature_cols))

print(f"Numeric features passing threshold (points > {MIN_SELECTION_POINTS}): {len(numeric_feature_cols)}")
print(f"Exception features (always retained)                           : {len(exception_cols)} ({n_new_exceptions} below threshold)")
print(f"Categorical features (always retained)                         : {len(cat_cols)}")
print(f"Time indicator features                                        : {len(time_indicator_cols)}")
print(f"Total unique features selected                                 : {len(feature_cols)}")

Numeric features passing threshold (points > 1): 439
Categorical features (always retained)                         : 10
Time indicator features                                        : 32
Total unique features selected                                 : 481


C:\Users\domjo\AppData\Local\Temp\ipykernel_30308\3492365585.py:13: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  feature_cols = pd.unique(numeric_feature_cols + cat_cols + time_indicator_cols).tolist()


In [8]:
pd.Series(feature_cols).to_csv(PATH_OUTPUT_FEATURES, index=False)
print(f"Saved selected feature list → {PATH_OUTPUT_FEATURES}")

pd.Series(exception_cols, name='feature').to_csv(PATH_OUTPUT_EXCEPTIONS, index=False)
print(f"Saved exception feature list ({len(exception_cols)}) → {PATH_OUTPUT_EXCEPTIONS}")

Saved selected feature list → a:\Dominic\My Projects\Home Credit\data\Model-Training\ModelDataset\selected_features.csv


## 4. Check for Duplicate Columns

Stops if the join produced two columns with the same name, or a feature is
listed twice. The last cell counts columns with pandas' `_x`/`_y` suffixes,
which would mean two tables had a column with the same name (should be 0).

In [9]:
dup_cols = combined_df.columns[combined_df.columns.duplicated()].unique().tolist()
if dup_cols:
    for col in dup_cols:
        matches = [c for c in combined_df.columns if c == col]
        print(f"  {col}: {len(matches)} copies")
    raise ValueError(f"Duplicate columns found in combined_df: {dup_cols}")
else:
    print("No duplicated combined_df columns detected.")

feature_dup_mask = pd.Series(feature_cols).duplicated(keep=False)
duped_features = pd.Series(feature_cols)[feature_dup_mask].unique().tolist()
if duped_features:
    raise ValueError(f"Duplicate feature names in feature_cols: {duped_features}")
else:
    print("No duplicated feature names in feature_cols.")

No duplicated combined_df columns detected.
No duplicated feature names in feature_cols.


In [10]:
sum(["_x" in col or "_y" in col for col in combined_df.columns if col not in join_keys])

0

## 5. Mark Text Columns as Categories

Text columns (e.g. `ORGANIZATION_TYPE`) are converted to pandas' `category`
type, which LightGBM and XGBoost can use directly.

The list of possible values for each comes from the whole dataset, before
any train/test split. That's normal for tree models and leaks nothing about
the target — it only fixes which values exist. It does mean a value never
seen here (e.g. a new organisation type at scoring time) needs handling
when the model is used.

In [11]:
for col in cat_cols:
    combined_df[col] = combined_df[col].astype('category')

print(f"Cast {len(cat_cols)} columns to category dtype.")

Cast 10 columns to category dtype.


## 6. Quality Checks and Save

Checks `TARGET` is present and there's one row per applicant, then saves
`SK_ID_CURR`, the kept features and `TARGET`. Parquet keeps the `category`
types, so the next notebooks see the same types.

In [12]:
assert 'TARGET' in combined_df.columns, "TARGET column missing from combined_df!"
n_dupes = combined_df['SK_ID_CURR'].duplicated().sum()
assert n_dupes == 0, f"Duplicate SK_ID_CURR in final dataset: {n_dupes}"

print(f"Final dataset shape : {combined_df.shape}")
print(f"Features            : {len(feature_cols)}")
print(f"Target distribution :\n{combined_df['TARGET'].value_counts(normalize=True).to_string()}")

Final dataset shape : (307511, 1229)
Features            : 481
Target distribution :
TARGET
0    0.919271
1    0.080729


In [13]:
model_dataset = combined_df[['SK_ID_CURR'] + feature_cols + ['TARGET']]
model_dataset.to_parquet(PATH_OUTPUT_DATASET, index=False)
print(f"Saved → {PATH_OUTPUT_DATASET}")

Saved → a:\Dominic\My Projects\Home Credit\data\Model-Training\ModelDataset\model_dataset_initial.parquet


## 7. Free Memory

In [14]:
del combined_df, model_dataset
import gc; gc.collect()
print("All DataFrames released.")

All DataFrames released.
